# People & platform profile page generator

This notebook reads `../data/smmh.csv` and creates a separate storytelling HTML page:

`../html-files/people.html`

It answers:
- What other apps TikTok users use
- How old users are, their affiliations, occupations, and relationship status
- Which apps are popular among people aged 60+
- Which apps are common among people spending **less than 1 hour/day** on social media
- Outliers: respondents with extremely high mental-health scores, the youngest respondent(s), and the oldest respondent(s)
- How non-users score across the mental-health questions, shown as a **small-n descriptive analysis** so it is not overinterpreted


In [49]:
import pandas as pd
import json
from pathlib import Path

# -----------------------------
# INPUT / OUTPUT
# -----------------------------
csv_file = "../data/smmh.csv"
out_html = "../html-files/others.html"

Path(out_html).parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_file)

# -----------------------------
# COLUMN DEFINITIONS
# -----------------------------
age_col = "1. What is your age?"
gender_col = "2. Gender"
relationship_col = "3. Relationship Status"
occupation_col = "4. Occupation Status"
affiliation_col = "5. What type of organizations are you affiliated with?"
use_col = "6. Do you use social media?"
platform_col = "7. What social media platforms do you commonly use?"
time_col = "8. What is the average time you spend on social media every day?"

question_cols = {
    "Q9": "9. How often do you find yourself using Social media without a specific purpose?",
    "Q10": "10. How often do you get distracted by Social media when you are busy doing something?",
    "Q11": "11. Do you feel restless if you haven't used Social media in a while?",
    "Q12": "12. On a scale of 1 to 5, how easily distracted are you?",
    "Q13": "13. On a scale of 1 to 5, how much are you bothered by worries?",
    "Q14": "14. Do you find it difficult to concentrate on things?",
    "Q15": "15. On a scale of 1-5, how often do you compare yourself to other successful people through the use of social media?",
    "Q16": "16. Following the previous question, how do you feel about these comparisons, generally speaking?",
    "Q17": "17. How often do you look to seek validation from features of social media?",
    "Q18": "18. How often do you feel depressed or down?",
    "Q19": "19. On a scale of 1 to 5, how frequently does your interest in daily activities fluctuate?",
    "Q20": "20. On a scale of 1 to 5, how often do you face issues regarding sleep?"
}

question_labels = {
    "Q9": "Purposeless use",
    "Q10": "Distracted while busy",
    "Q11": "Restless offline",
    "Q12": "Easily distracted",
    "Q13": "Bothered by worries",
    "Q14": "Concentration issues",
    "Q15": "Social comparison",
    "Q16": "Feeling after comparison",
    "Q17": "Validation seeking",
    "Q18": "Feeling down",
    "Q19": "Interest fluctuation",
    "Q20": "Sleep issues"
}

platforms = [
    "Instagram", "Facebook", "Twitter", "YouTube", "Discord",
    "Pinterest", "TikTok", "Snapchat", "Reddit", "LinkedIn"
]

# -----------------------------
# CLEANING HELPERS
# -----------------------------
df[age_col] = pd.to_numeric(df[age_col], errors="coerce")

for _, col in question_cols.items():
    df[col] = pd.to_numeric(df[col], errors="coerce")

def has_platform(series, platform):
    return series.fillna("").str.contains(platform, case=False, regex=False)

def value_counts_payload(series, top_n=None):
    vc = series.fillna("Unknown").value_counts()
    if top_n:
        vc = vc.head(top_n)

    total = int(vc.sum()) if int(vc.sum()) else 1

    return [
        {
            "label": str(k),
            "n": int(v),
            "pct": round(v / total * 100, 1)
        }
        for k, v in vc.items()
    ]

def multi_platform_counts(data):
    rows = []

    for p in platforms:
        n = int(has_platform(data[platform_col], p).sum())

        if n > 0:
            rows.append({
                "platform": p,
                "n": n,
                "pct": round(n / max(len(data), 1) * 100, 1)
            })

    return sorted(rows, key=lambda x: x["n"], reverse=True)

def time_sort_key(label):
    order = {
        "Less than an Hour": 0,
        "Less than an hour": 0,
        "Between 1 and 2 hours": 1,
        "Between 2 and 3 hours": 2,
        "Between 3 and 4 hours": 3,
        "Between 4 and 5 hours": 4,
        "More than 5 hours": 5,
    }
    return order.get(str(label), 99)

# -----------------------------
# SECTION 2: TIKTOK USERS ONLY
# -----------------------------
tiktok_users = df[has_platform(df[platform_col], "TikTok")].copy()

tiktok_other_apps = []

for app in platforms:
    if app == "TikTok":
        continue

    n = int(has_platform(tiktok_users[platform_col], app).sum())
    pct = round(n / max(len(tiktok_users), 1) * 100, 1)

    tiktok_other_apps.append({
        "label": app,
        "platform": app,
        "n": n,
        "pct": pct
    })

tiktok_other_apps = sorted(tiktok_other_apps, key=lambda x: x["n"], reverse=True)

tiktok_payload = {
    "n": int(len(tiktok_users)),
    "ageMean": round(tiktok_users[age_col].mean(), 1) if len(tiktok_users) else None,
    "occupation": value_counts_payload(tiktok_users[occupation_col]),
    "relationship": value_counts_payload(tiktok_users[relationship_col]),
    "otherApps": tiktok_other_apps
}

# -----------------------------
# SECTION 3: LIGHT USERS
# -----------------------------
light_users = df[
    df[time_col]
    .fillna("")
    .str.contains("Less than", case=False, regex=False)
].copy()

age_bins = [0, 17, 24, 34, 44, 59, 200]
age_labels = ["≤17", "18–24", "25–34", "35–44", "45–59", "60+"]

light_users["age_bin"] = pd.cut(
    light_users[age_col],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

light_age_dist = light_users["age_bin"].value_counts().reindex(age_labels, fill_value=0)

light_age_payload = [
    {"ageGroup": str(k), "n": int(v)}
    for k, v in light_age_dist.items()
]

light_gender_payload = value_counts_payload(light_users[gender_col])
light_occupation_payload = value_counts_payload(light_users[occupation_col])
light_app_payload = multi_platform_counts(light_users)

# -----------------------------
# SECTION 4: 60+ USERS
# -----------------------------
older = df[df[age_col] >= 60].copy()

older_time = (
    older[time_col]
    .fillna("Unknown")
    .value_counts()
    .sort_index(key=lambda idx: [time_sort_key(x) for x in idx])
)

older_payload = {
    "n": int(len(older)),
    "ageMean": round(older[age_col].mean(), 1) if len(older) else None,
    "ageMin": int(older[age_col].min()) if len(older) else None,
    "ageMax": int(older[age_col].max()) if len(older) else None,
    "apps": multi_platform_counts(older),
    "time": [
        {
            "label": str(k),
            "n": int(v),
            "pct": round(v / max(len(older), 1) * 100, 1)
        }
        for k, v in older_time.items()
    ]
}

# -----------------------------
# SECTION 5: OUTLIERS / EXTREMES
# -----------------------------
score_cols = list(question_cols.values())

df["mean_mh_score"] = df[score_cols].mean(axis=1)
df["n_5_scores"] = (df[score_cols] == 5).sum(axis=1)

perfect = df[df["mean_mh_score"] == 5].copy()

perfect_rows = []

for idx, row in perfect.iterrows():
    perfect_rows.append({
        "row": int(idx),
        "age": None if pd.isna(row[age_col]) else float(row[age_col]),
        "gender": str(row.get(gender_col, "Unknown")),
        "occupation": str(row.get(occupation_col, "Unknown")),
        "relationship": str(row.get(relationship_col, "Unknown")),
        "platforms": str(row.get(platform_col, "")),
        "time": str(row.get(time_col, "")),
        "meanScore": round(float(row["mean_mh_score"]), 2),
        "nFive": int(row["n_5_scores"])
    })

age_valid = df[df[age_col].notna()].copy()

youngest = age_valid.loc[age_valid[age_col].idxmin()]
oldest = age_valid.loc[age_valid[age_col].idxmax()]

def person_payload(row):
    return {
        "row": int(row.name),
        "age": float(row[age_col]),
        "gender": str(row.get(gender_col, "Unknown")),
        "occupation": str(row.get(occupation_col, "Unknown")),
        "relationship": str(row.get(relationship_col, "Unknown")),
        "affiliation": str(row.get(affiliation_col, "Unknown")),
        "usesSocial": str(row.get(use_col, "Unknown")),
        "platforms": str(row.get(platform_col, "")),
        "time": str(row.get(time_col, "")),
        "meanScore": round(float(row[score_cols].mean()), 2) if row[score_cols].notna().any() else None
    }

outlier_payload = {
    "perfectScorePeople": perfect_rows,
    "perfectCount": int(len(perfect_rows)),
    "youngest": person_payload(youngest),
    "oldest": person_payload(oldest)
}

# -----------------------------
# SECTION 6: NON-USERS
# -----------------------------
non_users = df[
    df[use_col]
    .fillna("")
    .str.lower()
    .str.strip()
    .eq("no")
].copy()

non_user_scores = []

for q, col in question_cols.items():
    non_user_scores.append({
        "question": question_labels[q],
        "full": col,
        "score": round(non_users[col].mean(), 2) if len(non_users) else None
    })

non_user_payload = {
    "n": int(len(non_users)),
    "scores": non_user_scores,
    "people": [person_payload(row) for _, row in non_users.iterrows()]
}

# -----------------------------
# FINAL PAYLOAD
# -----------------------------
payload = {
    "tiktok": tiktok_payload,
    "lightUsers": {
        "n": int(len(light_users)),
        "ageDistribution": light_age_payload,
        "genderDistribution": light_gender_payload,
        "occupationDistribution": light_occupation_payload,
        "apps": light_app_payload
    },
    "older60": older_payload,
    "outliers": outlier_payload,
    "nonUsers": non_user_payload
}

DATA_JSON = json.dumps(payload, indent=2)

# -----------------------------
# HTML TEMPLATE
# -----------------------------
html = r'''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Who Are Extremes?</title>
<meta name="viewport" content="width=device-width, initial-scale=1.0">

<link href="https://fonts.googleapis.com/css2?family=Playfair+Display:ital,wght@0,400;0,700;0,900;1,400;1,700&family=DM+Mono:wght@300;400;500&family=DM+Sans:wght@300;400;500&display=swap" rel="stylesheet">

<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>

<style>
:root {
  --cream:#F5F0E8;
  --dark:#1A1209;
  --warm:#C8A96E;
  --accent:#E8472A;
  --muted:#6B5E4A;
  --card:#FFFDF7;
  --border:rgba(200,169,110,0.28);
}

* {
  box-sizing:border-box;
  margin:0;
  padding:0;
}

body {
  background:var(--cream);
  color:var(--dark);
  font-family:'DM Sans',sans-serif;
  font-weight:300;
  line-height:1.7;
  overflow-x:hidden;
}

.hero {
  min-height:72vh;
  display:flex;
  flex-direction:column;
  justify-content:center;
  align-items:center;
  text-align:center;
  padding:4rem 2rem;
  position:relative;
  overflow:hidden;
}

.hero::before {
  content:'';
  position:absolute;
  inset:0;
  background:
    repeating-linear-gradient(0deg,transparent,transparent 39px,rgba(200,169,110,.11) 39px,rgba(200,169,110,.11) 40px),
    repeating-linear-gradient(90deg,transparent,transparent 39px,rgba(200,169,110,.11) 39px,rgba(200,169,110,.11) 40px);
  pointer-events:none;
}

.eyebrow {
  font-family:'DM Mono',monospace;
  font-size:11px;
  letter-spacing:.22em;
  text-transform:uppercase;
  color:var(--muted);
  margin-bottom:2rem;
  position:relative;
}

.hero h1 {
  font-family:'Playfair Display',serif;
  font-size:clamp(3rem,8vw,6.5rem);
  font-weight:900;
  line-height:1;
  max-width:950px;
  position:relative;
}

.hero h1 em {
  color:var(--accent);
  font-style:italic;
}

.hero-sub {
  max-width:620px;
  color:var(--muted);
  margin-top:1.75rem;
  position:relative;
}

section {
  max-width:1100px;
  margin:0 auto;
  padding:5rem 2rem;
}

.sec-label {
  font-family:'DM Mono',monospace;
  font-size:10px;
  letter-spacing:.25em;
  text-transform:uppercase;
  color:var(--warm);
  margin-bottom:.9rem;
}

h2 {
  font-family:'Playfair Display',serif;
  font-size:clamp(2rem,4vw,3rem);
  line-height:1.15;
  margin-bottom:.9rem;
}

.lead {
  max-width:720px;
  color:var(--muted);
  margin-bottom:2rem;
}

.card {
  background:var(--card);
  border:1px solid var(--border);
  border-radius:18px;
  padding:2rem;
  margin-bottom:1.5rem;
  box-shadow:0 10px 30px rgba(26,18,9,.04);
}

.dark-card {
  background:var(--dark);
  color:var(--cream);
  border-radius:24px;
  padding:2.4rem;
  border:1px solid rgba(200,169,110,.25);
  box-shadow:0 18px 45px rgba(26,18,9,.18);
  margin-bottom:1.5rem;
}

.grid-2 {
  display:grid;
  grid-template-columns:1fr 1fr;
  gap:1.5rem;
}

.grid-3 {
  display:grid;
  grid-template-columns:repeat(3,1fr);
  gap:1rem;
}

.chart-wrap {
  position:relative;
  height:330px;
}

.chart-small {
  height:270px;
}

.stat {
  background:rgba(245,240,232,.06);
  border:1px solid rgba(200,169,110,.18);
  border-radius:16px;
  padding:1.2rem;
}

.stat-num {
  font-family:'Playfair Display',serif;
  font-size:2.4rem;
  font-weight:900;
  color:var(--warm);
  display:block;
  line-height:1;
}

.stat-label {
  font-family:'DM Mono',monospace;
  font-size:9px;
  letter-spacing:.15em;
  text-transform:uppercase;
  color:rgba(245,240,232,.55);
  margin-top:.5rem;
}

.table {
  width:100%;
  border-collapse:collapse;
  font-family:'DM Mono',monospace;
  font-size:11px;
}

.table th,
.table td {
  border-bottom:1px solid var(--border);
  padding:.75rem .45rem;
  text-align:left;
}

.table th {
  color:var(--muted);
  text-transform:uppercase;
  font-size:9px;
  letter-spacing:.12em;
}

.person-card {
  background:var(--dark);
  color:var(--cream);
  border-radius:20px;
  padding:1.6rem;
  border:1px solid rgba(200,169,110,.25);
  position:relative;
  overflow:hidden;
}

.person-card::after {
  content:attr(data-age);
  position:absolute;
  right:1rem;
  top:-.6rem;
  font-family:'Playfair Display',serif;
  font-weight:900;
  font-size:5rem;
  color:rgba(200,169,110,.11);
  line-height:1;
}

.person-title {
  font-family:'Playfair Display',serif;
  font-size:1.6rem;
  font-weight:900;
  margin-bottom:.35rem;
}

.person-meta {
  color:rgba(245,240,232,.7);
  font-size:.9rem;
  max-width:85%;
}

.person-detail {
  font-family:'DM Mono',monospace;
  font-size:10px;
  color:rgba(245,240,232,.55);
  margin-top:1rem;
  line-height:1.7;
}

.severity-grid {
  display:grid;
  grid-template-columns:repeat(auto-fit,minmax(230px,1fr));
  gap:14px;
}

.severity-card {
  background:linear-gradient(135deg,rgba(232,71,42,.18),rgba(200,169,110,.10));
  border:1px solid rgba(232,71,42,.25);
  border-radius:18px;
  padding:1.4rem;
  min-height:180px;
}

.severity-title {
  font-family:'Playfair Display',serif;
  font-size:1.4rem;
  font-weight:900;
}

.severity-score {
  font-family:'Playfair Display',serif;
  font-size:2.4rem;
  font-weight:900;
  color:var(--accent);
  line-height:1;
  margin:.8rem 0;
}

.severity-meta {
  font-family:'DM Mono',monospace;
  font-size:10px;
  color:var(--muted);
  line-height:1.7;
}

.note {
  font-size:.9rem;
  color:var(--muted);
  background:rgba(200,169,110,.12);
  border-left:3px solid var(--warm);
  padding:1rem;
  border-radius:10px;
  margin-bottom:1.5rem;
}

.toggle-row {
  display:flex;
  flex-wrap:wrap;
  gap:.5rem;
  margin-bottom:1rem;
}

.toggle-btn {
  border:1px solid var(--border);
  background:rgba(255,253,247,.8);
  color:var(--dark);
  border-radius:999px;
  padding:.45rem .75rem;
  font-family:'DM Mono',monospace;
  font-size:10px;
  letter-spacing:.1em;
  text-transform:uppercase;
  cursor:pointer;
}

.toggle-btn.active {
  background:var(--dark);
  color:var(--cream);
}

@media(max-width:780px) {
  .grid-2,
  .grid-3 {
    grid-template-columns:1fr;
  }

  .chart-wrap {
    height:300px;
  }
}
</style>
</head>

<body>

<div class="hero">
  <p class="eyebrow">A closer look at the respondents</p>
  <h1>Let’s look at the <em>extremes</em></h1>
  <p class="hero-sub">
    Want to know what else the TikTokers are doing? Where are the light scrollers?
    How are the non-users doing mentally? And what are the other outliers?
    This page shows the profiles of these different subgroups.
  </p>
</div>

<section id="tiktok">
  <p class="sec-label">02 — TikTok users</p>
  <h2>Who are the TikTok users?</h2>
  <p class="lead">
    This section only counts respondents who reported using TikTok. The plots show their occupation,
    relationship status, and which other apps are popular among them.
  </p>

  <div class="dark-card">
    <div class="grid-3" id="tiktokStats"></div>
  </div>

  <div class="grid-2">
    <div class="card">
      <h3>Occupation among TikTok users</h3>
      <div class="chart-wrap chart-small">
        <canvas id="tiktokOccupationPie"></canvas>
      </div>
    </div>

    <div class="card">
      <h3>Relationship status among TikTok users</h3>
      <div class="chart-wrap chart-small">
        <canvas id="tiktokRelationshipPie"></canvas>
      </div>
    </div>
  </div>

  <div class="card">
    <h3>What other apps are popular among TikTok users?</h3>
    <p class="note">
      Respondents could select multiple platforms. These values show how many TikTok users also use each app.
    </p>
    <div class="chart-wrap">
      <canvas id="tiktokOtherAppsChart"></canvas>
    </div>
  </div>
</section>

<section id="light">
  <p class="sec-label">03 — Light users</p>
  <h2>Who spends less than one hour online?</h2>
  <p class="lead">
    This section focuses only on people reporting less than one hour of social media per day.
  </p>

  <div class="grid-2">
    <div class="card">
      <h3>Who are the light users?</h3>
      <p class="note">
        Choose whether to view light users by age, gender, or occupation.
      </p>

      <div class="toggle-row">
        <button class="toggle-btn active" onclick="updateLightProfileChart('age', this)">Age</button>
        <button class="toggle-btn" onclick="updateLightProfileChart('gender', this)">Gender</button>
        <button class="toggle-btn" onclick="updateLightProfileChart('occupation', this)">Occupation</button>
      </div>

      <div class="chart-wrap">
        <canvas id="lightProfileChart"></canvas>
      </div>
    </div>

    <div class="card">
      <h3>Apps used by light users</h3>
      <div class="chart-wrap">
        <canvas id="lightAppsChart"></canvas>
      </div>
    </div>
  </div>
</section>

<section id="older">
  <p class="sec-label">04 — Older respondents</p>
  <h2>What about people aged 60+?</h2>
  <p class="lead">
    For older respondents, the page reports the most used apps and how much time they spend on social media.
  </p>

  <div class="dark-card">
    <div class="grid-3" id="olderStats"></div>
  </div>

  <div class="grid-2">
    <div class="card">
      <h3>Most used apps among 60+</h3>
      <div class="chart-wrap">
        <canvas id="olderAppsChart"></canvas>
      </div>
    </div>

    <div class="card">
      <h3>Daily time among 60+</h3>
      <div class="chart-wrap">
        <canvas id="olderTimeChart"></canvas>
      </div>
    </div>
  </div>
</section>

<section id="outliers">
  <p class="sec-label">05 — Outliers and extremes</p>
  <h2>The most extreme profiles.</h2>
  <p class="lead">
    This section shows respondents with a perfect 5/5 average across all mental-health questions,
    plus the youngest and oldest respondent profiles.
  </p>

  <div class="card">
    <h3>Most severe health scores</h3>
    <p class="note">
      Only respondents with an average score of exactly 5/5 across questions 9–20 are shown here.
    </p>
    <div class="severity-grid" id="severityCards"></div>
  </div>

  <div class="grid-2" id="ageExtremeCards"></div>
</section>

<section id="nonusers">
  <p class="sec-label">06 — Non-users</p>
  <h2>The last free souls or liars</h2>
  <p class="lead">
    Three respondents reported not using social media at all. This section keeps them separate, so their tiny sample size is clear.
  </p>

  <div class="card">
    <h3>Non-user respondent table</h3>
    <table class="table" id="nonUserTable"></table>
  </div>

  <div class="card">
    <h3>Non-user scores across mental-health questions</h3>
    <div class="chart-wrap">
      <canvas id="nonUserChart"></canvas>
    </div>
  </div>

  <div class="dark-card">
    <p id="nonUserIntro"></p>
  </div>
</section>

<script>
const D = __DATA_JSON__;

const COLORS = [
  "#E8472A", "#C8A96E", "#534AB7", "#185FA5", "#D4537E",
  "#3B6D11", "#BA7517", "#888780", "#0F6E56", "#A64253"
];

Chart.defaults.font.family = "'DM Mono', monospace";
Chart.defaults.color = "#6B5E4A";

function makePie(id, rows) {
  const el = document.getElementById(id);
  if (!el) return;

  new Chart(el, {
    type: "doughnut",
    data: {
      labels: rows.map(x => x.label),
      datasets: [{
        data: rows.map(x => x.n),
        backgroundColor: COLORS,
        borderColor: "#FFFDF7",
        borderWidth: 3
      }]
    },
    options: {
      responsive: true,
      maintainAspectRatio: false,
      cutout: "58%",
      plugins: {
        legend: {
          position: "bottom",
          labels: {
            boxWidth: 10,
            font: { size: 10 }
          }
        },
        tooltip: {
          callbacks: {
            label: c => ` ${c.label}: ${c.parsed} (${rows[c.dataIndex].pct}%)`
          }
        }
      }
    }
  });
}

/* -----------------------------
   SECTION 02: TIKTOK USERS
----------------------------- */
document.getElementById("tiktokStats").innerHTML = `
  <div class="stat">
    <span class="stat-num">${D.tiktok.n}</span>
    <div class="stat-label">TikTok users</div>
  </div>
  <div class="stat">
    <span class="stat-num">${D.tiktok.ageMean ?? "—"}</span>
    <div class="stat-label">Average age</div>
  </div>
  <div class="stat">
    <span class="stat-num">${D.tiktok.otherApps[0]?.label ?? "—"}</span>
    <div class="stat-label">Most common other app</div>
  </div>
`;

makePie("tiktokOccupationPie", D.tiktok.occupation);
makePie("tiktokRelationshipPie", D.tiktok.relationship);

new Chart(document.getElementById("tiktokOtherAppsChart"), {
  type: "bar",
  data: {
    labels: D.tiktok.otherApps.map(x => x.label),
    datasets: [{
      data: D.tiktok.otherApps.map(x => x.n),
      backgroundColor: "#E8472Abb",
      borderColor: "#E8472A",
      borderWidth: 1
    }]
  },
  options: {
    indexAxis: "y",
    responsive: true,
    maintainAspectRatio: false,
    plugins: {
      legend: { display: false },
      tooltip: {
        callbacks: {
          label: c => `${c.parsed.x} TikTok users (${D.tiktok.otherApps[c.dataIndex].pct}%)`
        }
      }
    },
    scales: {
      x: {
        beginAtZero: true,
        ticks: { stepSize: 1, precision: 0 },
        title: {
          display: true,
          text: "Number of TikTok users also using this app"
        }
      },
      y: { grid: { display: false } }
    }
  }
});

/* -----------------------------
   SECTION 03: LIGHT USERS
----------------------------- */
let lightProfileChart = null;

function updateLightProfileChart(mode, btn = null) {
  if (btn) {
    document.querySelectorAll(".toggle-btn").forEach(b => b.classList.remove("active"));
    btn.classList.add("active");
  }

  let labels;
  let values;
  let xTitle;

  if (mode === "age") {
    labels = D.lightUsers.ageDistribution.map(x => x.ageGroup);
    values = D.lightUsers.ageDistribution.map(x => x.n);
    xTitle = "Age group";
  } else if (mode === "gender") {
    labels = D.lightUsers.genderDistribution.map(x => x.label);
    values = D.lightUsers.genderDistribution.map(x => x.n);
    xTitle = "Gender";
  } else {
    labels = D.lightUsers.occupationDistribution.map(x => x.label);
    values = D.lightUsers.occupationDistribution.map(x => x.n);
    xTitle = "Occupation";
  }

  if (lightProfileChart) {
    lightProfileChart.destroy();
  }

  lightProfileChart = new Chart(document.getElementById("lightProfileChart"), {
    type: "bar",
    data: {
      labels: labels,
      datasets: [{
        data: values,
        backgroundColor: "#E8472Abb",
        borderColor: "#E8472A",
        borderWidth: 1
      }]
    },
    options: {
      responsive: true,
      maintainAspectRatio: false,
      plugins: { legend: { display: false } },
      scales: {
        x: {
          grid: { display: false },
          ticks: {
            maxRotation: 35,
            minRotation: 20
          },
          title: { display: true, text: xTitle }
        },
        y: {
          beginAtZero: true,
          ticks: { stepSize: 1, precision: 0 },
          title: { display: true, text: "Number of light users" },
          grid: { color: "rgba(200,169,110,.15)" }
        }
      }
    }
  });
}

updateLightProfileChart("age");

new Chart(document.getElementById("lightAppsChart"), {
  type: "bar",
  data: {
    labels: D.lightUsers.apps.map(x => x.platform),
    datasets: [{
      data: D.lightUsers.apps.map(x => x.n),
      backgroundColor: "#C8A96Ebb",
      borderColor: "#C8A96E",
      borderWidth: 1
    }]
  },
  options: {
    indexAxis: "y",
    responsive: true,
    maintainAspectRatio: false,
    plugins: {
      legend: { display: false },
      tooltip: {
        callbacks: {
          label: c => `${c.parsed.x} users (${D.lightUsers.apps[c.dataIndex].pct}%)`
        }
      }
    },
    scales: {
      x: {
        beginAtZero: true,
        ticks: { stepSize: 1, precision: 0 },
        title: { display: true, text: "Number of light users" }
      },
      y: { grid: { display: false } }
    }
  }
});

/* -----------------------------
   SECTION 04: 60+ USERS
----------------------------- */
document.getElementById("olderStats").innerHTML = `
  <div class="stat">
    <span class="stat-num">${D.older60.n}</span>
    <div class="stat-label">Respondents aged 60+</div>
  </div>
  <div class="stat">
    <span class="stat-num">${D.older60.ageMean ?? "—"}</span>
    <div class="stat-label">Average age</div>
  </div>
  <div class="stat">
    <span class="stat-num">${D.older60.ageMin ?? "—"}–${D.older60.ageMax ?? "—"}</span>
    <div class="stat-label">Age range</div>
  </div>
`;

new Chart(document.getElementById("olderAppsChart"), {
  type: "bar",
  data: {
    labels: D.older60.apps.map(x => x.platform),
    datasets: [{
      data: D.older60.apps.map(x => x.n),
      backgroundColor: "#534AB7bb",
      borderColor: "#534AB7",
      borderWidth: 1
    }]
  },
  options: {
    indexAxis: "y",
    responsive: true,
    maintainAspectRatio: false,
    plugins: {
      legend: { display: false },
      tooltip: {
        callbacks: {
          label: c => `${c.parsed.x} users (${D.older60.apps[c.dataIndex].pct}%)`
        }
      }
    },
    scales: {
      x: {
        beginAtZero: true,
        ticks: { stepSize: 1, precision: 0 },
        title: { display: true, text: "Number of 60+ users" }
      },
      y: { grid: { display: false } }
    }
  }
});

new Chart(document.getElementById("olderTimeChart"), {
  type: "bar",
  data: {
    labels: D.older60.time.map(x => x.label),
    datasets: [{
      data: D.older60.time.map(x => x.n),
      backgroundColor: "#3B6D11bb",
      borderColor: "#3B6D11",
      borderWidth: 1
    }]
  },
  options: {
    responsive: true,
    maintainAspectRatio: false,
    plugins: {
      legend: { display: false },
      tooltip: {
        callbacks: {
          label: c => `${c.parsed.y} respondents (${D.older60.time[c.dataIndex].pct}%)`
        }
      }
    },
    scales: {
      x: {
        grid: { display: false },
        ticks: { maxRotation: 35, minRotation: 25 }
      },
      y: {
        beginAtZero: true,
        ticks: { stepSize: 1, precision: 0 },
        title: { display: true, text: "Number of 60+ respondents" }
      }
    }
  }
});

/* -----------------------------
   SECTION 05: OUTLIERS
----------------------------- */
const severity = D.outliers.perfectScorePeople;

document.getElementById("severityCards").innerHTML = severity.length
  ? severity.map(p => `
    <div class="severity-card">
      <div class="severity-title">Respondent ${p.row}</div>
      <div class="severity-score">5.00 / 5</div>
      <div class="severity-meta">
        Age: ${p.age ?? "Unknown"}<br>
        Occupation: ${p.occupation}<br>
        Relationship: ${p.relationship}<br>
        Platforms: ${p.platforms || "None listed"}<br>
        Daily time: ${p.time || "Unknown"}
      </div>
    </div>
  `).join("")
  : `
    <div class="severity-card">
      <div class="severity-title">No perfect 5/5 average</div>
      <div class="severity-score">0</div>
      <div class="severity-meta">
        No respondent averaged exactly 5/5 across all questions 9–20.
      </div>
    </div>
  `;

function personCard(title, p) {
  return `
    <div class="person-card" data-age="${p.age}">
      <div class="person-title">${title}</div>
      <div class="person-meta">${p.age} years old · ${p.gender} · ${p.relationship}</div>
      <div class="person-detail">
        Occupation: ${p.occupation}<br>
        Affiliation: ${p.affiliation}<br>
        Uses social media: ${p.usesSocial}<br>
        Platforms: ${p.platforms || "None listed"}<br>
        Daily time: ${p.time || "Unknown"}<br>
        Mean mental-health score: ${p.meanScore ?? "—"} / 5
      </div>
    </div>
  `;
}

document.getElementById("ageExtremeCards").innerHTML =
  personCard("Youngest respondent", D.outliers.youngest) +
  personCard("Oldest respondent", D.outliers.oldest);

/* -----------------------------
   SECTION 06: NON-USERS
----------------------------- */
document.getElementById("nonUserIntro").innerHTML = `
  These respondents should be interpreted as a tiny separate group, not as a comparison against all social media users.
  The sample contains only <strong>${D.nonUsers.n}</strong> non-users, so this is descriptive rather than conclusive.
`;

new Chart(document.getElementById("nonUserChart"), {
  type: "bar",
  data: {
    labels: D.nonUsers.scores.map(x => x.question),
    datasets: [{
      data: D.nonUsers.scores.map(x => x.score),
      backgroundColor: "#E8472Abb",
      borderColor: "#E8472A",
      borderWidth: 1
    }]
  },
  options: {
    indexAxis: "y",
    responsive: true,
    maintainAspectRatio: false,
    plugins: {
      legend: { display: false },
      tooltip: {
        callbacks: {
          label: c => `${c.parsed.x ?? "NA"} / 5`,
          afterLabel: c => D.nonUsers.scores[c.dataIndex].full
        }
      }
    },
    scales: {
      x: {
        min: 1,
        max: 5,
        title: { display: true, text: "Average score among non-users only" }
      },
      y: {
        grid: { display: false },
        title: { display: true, text: "Mental-health question" }
      }
    }
  }
});

document.getElementById("nonUserTable").innerHTML =
  `<tr>
    <th>Row</th>
    <th>Age</th>
    <th>Gender</th>
    <th>Occupation</th>
    <th>Relationship</th>
    <th>Mean score</th>
  </tr>` +
  D.nonUsers.people.map(p => `
    <tr>
      <td>${p.row}</td>
      <td>${p.age}</td>
      <td>${p.gender}</td>
      <td>${p.occupation}</td>
      <td>${p.relationship}</td>
      <td>${p.meanScore ?? "—"} / 5</td>
    </tr>
  `).join("");
</script>

</body>
</html>
'''

html = html.replace("__DATA_JSON__", DATA_JSON)

with open(out_html, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Saved HTML to: {out_html}")

Saved HTML to: ../html-files/others.html
